In [1]:
!pip install ultralytics optuna pandas
!pip install -q legacy-cgi opendatasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 143.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 257.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 197.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 132.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 266.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 183.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 MB 200.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 229.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [ultralytics] [ultralytics]n]-32]


In [2]:
import opendatasets as od
from pathlib import Path

PERSISTENT_DIR = '/workspace/Dataset'

if Path(PERSISTENT_DIR).is_dir():
    print('Dataset already on persistent volume. Skipping download...')
else:
    print('Downloading dataset to persistent /workspace...')
    od.download(
        'https://www.kaggle.com/datasets/raunakgola/kitchenware-object-detection-dataset-yolo-format',
        data_dir=PERSISTENT_DIR)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds


Your Kaggle username:  raunakgola
Your Kaggle Key:  ········


Dataset URL: https://www.kaggle.com/datasets/raunakgola/kitchenware-object-detection-dataset-yolo-format


100%|██████████| 1.57G/1.57G [00:51<00:00, 32.9MB/s]


In [3]:
import shutil
from pathlib import Path

LOCAL_DIR = '/root/Dataset'

if not Path(LOCAL_DIR).exists():
    print("Copying dataset to local container disk (one-time per pod)...")
    shutil.copytree(f'{PERSISTENT_DIR}/kitchenware-object-detection-dataset-yolo-format',
                     LOCAL_DIR)
    print("Copy complete.")
else:
    print("Local copy already exists.")

# Write the yaml pointing at the LOCAL copy, not /workspace
yaml_content = f"""path: {LOCAL_DIR}/Dataset
train: train/images
val: val/images
nc: 6
names: ["plate", "spoon", "fork", "knife", "cup", "glass"]
"""
with open('/root/kitchenware.yaml', 'w') as f:
    f.write(yaml_content)

print(open('/root/kitchenware.yaml').read())

Copying dataset to local container disk (one-time per pod)...
Copy complete.
path: /root/Dataset/Dataset
train: train/images
val: val/images
nc: 6
names: ["plate", "spoon", "fork", "knife", "cup", "glass"]



In [4]:
import os
import torch

!df -h /root /workspace
print()
print("CPUs:", os.cpu_count())
!free -h
print()
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# real kernel-execution test, not just detection
x = torch.randn(1024, 1024, device="cuda")
print("Matmul OK:", (x @ x).shape)

Filesystem      Size  Used Avail Use% Mounted on
overlay          30G  4.0G   27G  14% /
overlay          30G  4.0G   27G  14% /

CPUs: 48
               total        used        free      shared  buff/cache   available
Mem:           251Gi        97Gi        58Gi        56Mi        96Gi       153Gi
Swap:             0B          0B          0B

Torch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
Matmul OK: torch.Size([1024, 1024])


In [5]:
import gc
import time
import torch
from ultralytics import YOLO

DATA_YAML = "/root/kitchenware.yaml"
CALIB_MODELS = ["yolo26n.pt", "yolo26s.pt", "yolov8n.pt", "yolov8s.pt"]  # one small sample, not all 10 — quick check

for m in CALIB_MODELS:
    model = None
    try:
        model = YOLO(m)
        t0 = time.time()
        model.train(data=DATA_YAML, epochs=2, imgsz=640, batch=16, workers=12,
                    cache="disk", device=0, verbose=False, plots=False,
                    project="runs/calib_4090", name=m.replace(".pt", ""), exist_ok=True)
        print(f"{m}: {time.time() - t0:.1f}s total for 2 epochs")
    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.177 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/kitchenware.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1331: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  attn = ((q * self.scale).transpose(-2, -1) @ k).softmax(dim=-1)
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1332: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is 

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4601.5±1824.6 MB/s, size: 160.9 KB)
train: Scanning /root/Dataset/Dataset/train/labels... 7187 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7187/7187 3.6Kit/s 2.0s0.0s
train: New cache created: /root/Dataset/Dataset/train/labels.cache
train: Caching images (14.3GB Disk): 100% ━━━━━━━━━━━━ 7187/7187 3.2Kit/s 2.3s0.1s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4379.0±1375.4 MB/s, size: 159.3 KB)
val: Scanning /root/Dataset/Dataset/val/labels... 1797 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1797/1797 3.2Kit/s 0.6s0.1s
val: New cache created: /root/Dataset/Dataset/val/labels.cache
val: Caching images (3.6GB Disk): 100% ━━━━━━━━━━━━ 1797/1797 3.1Kit/s 0.6s0.1s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and determining best 'optimizer' and 'lr0' automatically... 
optimizer: AdamW(lr=0.001, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(d

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1331: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  attn = ((q * self.scale).transpose(-2, -1) @ k).softmax(dim=-1)
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1332: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is 


      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        1/2      2.51G     0.8889      25.14     0.0215         16        640: 0% ──────────── 3/450 8.4it/s 4.7s<53.1s

...

WARNING ⚠️ train: 16.0GB disk space required, with 10% safety margin but only 8.0/30.0GB free, not caching images to disk
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5343.1±1594.4 MB/s, size: 159.3 KB)
val: Scanning /root/Dataset/Dataset/val/labels.cache... 1797 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1797/1797 837.5Mit/s 0.0s
val: Caching images (3.6GB Disk): 100% ━━━━━━━━━━━━ 1797/1797 22.6Kit/s 0.1s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and determining best 'optimizer' and 'lr0' automatically... 
optimizer: AdamW(lr=0.001, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Using 7187 train, 1797 val images for fraction=1.0 at imgsz=640
Using 12 dataloader workers
Logging results to /workspace/runs/detect/runs/calib_4090/yolov8s
Starting training for 2 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/2      3.23G      1.153 

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py:407: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


        1/2      3.52G     0.7448      1.743      1.184          3        640: 100% ━━━━━━━━━━━━ 450/450 26.2it/s 17.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 13.1it/s 4.3s0.0s
                   all       1797       1820      0.634      0.547      0.582      0.382

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/2       4.8G     0.7863       1.19      1.147         20        640: 0% ──────────── 2/450 5.4it/s 0.1s<1:22

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py:407: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


        2/2       4.8G     0.6156     0.7428      1.072          3        640: 100% ━━━━━━━━━━━━ 450/450 27.7it/s 16.3s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 23.7it/s 2.4s0.1s
                   all       1797       1820      0.906      0.863      0.934      0.814

2 epochs completed in 0.011 hours.
Optimizer stripped from /workspace/runs/detect/runs/calib_4090/yolov8s/weights/last.pt, 22.5MB
Optimizer stripped from /workspace/runs/detect/runs/calib_4090/yolov8s/weights/best.pt, 22.5MB

Validating /workspace/runs/detect/runs/calib_4090/yolov8s/weights/best.pt...
Ultralytics 8.4.177 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 25.1it/s 2.3s0.1s
                   all       1797       

In [ ]:
"""
Bayesian hyperparameter + architecture search using Optuna's TPE sampler
(Tree-structured Parzen Estimator — this IS the Bayesian optimization
algorithm, made explicit below) with median pruning to cut weak trials short.

Fixes applied vs. the earlier RunPod run:
- Trains from LOCAL disk (/root), not the network-mounted /workspace volume
  that caused the slow I/O + disk-quota crashes.
- Fixed batch list instead of AutoBatch, capped at 32 — the 5090 log showed
  a real CUDA OOM at batch=32 during AutoBatch's own probing for yolo12s
  (9.25M params), meaning attention-based blocks (A2C2f, C2PSA in
  yolo12/yolo26) use more VRAM per sample than plain conv nets like yolov8.
  With only 24GB on the 4090 (vs the 5090's 32GB), going above 32 risks
  the same crash.
- try/finally cleanup so a crashed trial doesn't leak memory into the next.
"""

import gc
import os
import time

import optuna
import pandas as pd
import torch
from ultralytics import YOLO

DATA_YAML = "/root/kitchenware.yaml"
IMG_SIZE = 640
MAX_EPOCHS = 15
N_TRIALS = 25
WORKERS = 12                     # leaves headroom on the 16 vCPU pod
PROJECT_DIR = "runs/optuna_4090"
BATCH_OPTIONS = [8, 16, 32]      # capped for 24GB VRAM safety — see note above

CANDIDATE_MODELS = [
    "yolo26n.pt", "yolo26s.pt", "yolo12n.pt", "yolo12s.pt",
    "yolo11n.pt", "yolo11s.pt", "yolov9t.pt", "yolov9s.pt",
    "yolov8n.pt", "yolov8s.pt",
]
MAP_METRIC_KEY = "metrics/mAP50-95(B)"


def objective(trial: optuna.Trial) -> float:
    model_name = trial.suggest_categorical("model", CANDIDATE_MODELS)
    lr0 = trial.suggest_float("lr0", 1e-4, 1e-2, log=True)
    batch = trial.suggest_categorical("batch", BATCH_OPTIONS)
    mosaic = trial.suggest_float("mosaic", 0.0, 1.0)

    model = None
    t0 = time.time()
    try:
        model = YOLO(model_name)

        def report_epoch(trainer):
            map5095 = trainer.metrics.get(MAP_METRIC_KEY, 0.0)
            trial.report(map5095, step=trainer.epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        model.add_callback("on_fit_epoch_end", report_epoch)

        model.train(
            data=DATA_YAML, epochs=MAX_EPOCHS, imgsz=IMG_SIZE, batch=batch,
            lr0=lr0, optimizer="AdamW", mosaic=mosaic, project=PROJECT_DIR,
            name=f"trial_{trial.number}", exist_ok=True, verbose=False,
            patience=15, plots=False, cache="disk", workers=WORKERS, device=0,
        )

        metrics = model.val(data=DATA_YAML, imgsz=IMG_SIZE, verbose=False)
        map5095 = float(metrics.box.map)

        trial.set_user_attr("model_name", model_name)
        trial.set_user_attr("map50", float(metrics.box.map50))
        trial.set_user_attr("inference_ms", float(metrics.speed.get("inference", 0.0)))
        trial.set_user_attr("trial_seconds", round(time.time() - t0, 1))

        return map5095
    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()


if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        study_name="yolo_mobile_screen_4090",
        sampler=optuna.samplers.TPESampler(),       # explicit Bayesian optimization
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=N_TRIALS)

    df = study.trials_dataframe()
    df.to_csv("/root/optuna_screening_results.csv", index=False)

    print("\nBest trial:", study.best_trial.params, study.best_trial.value)
    cols = ["number", "value", "state", "user_attrs_model_name", "user_attrs_trial_seconds"]
    print(df[cols].sort_values("value", ascending=False).head(10).to_string(index=False))

[I 2026-10-14 05:27:08,629] A new study created in memory with name: yolo_mobile_screen_4090


Ultralytics 8.4.177 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/kitchenware.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.008364964341397881, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9t.pt, momentum=0.937, mosaic=0.8920683957807084, multi_scale=0.0, name=trial_0, nbs=64, nms=None

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1331: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  attn = ((q * self.scale).transpose(-2, -1) @ k).softmax(dim=-1)
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1332: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is 

WARNING ⚠️ train: 16.0GB disk space required, with 10% safety margin but only 7.9/30.0GB free, not caching images to disk
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5257.6±1657.9 MB/s, size: 159.3 KB)
val: Scanning /root/Dataset/Dataset/val/labels.cache... 1797 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1797/1797 942.1Mit/s 0.0s
val: Caching images (3.6GB Disk): 100% ━━━━━━━━━━━━ 1797/1797 20.6Kit/s 0.1s
optimizer: AdamW(lr=0.008364964341397881, momentum=0.937) with parameter groups 221 weight(decay=0.0), 228 weight(decay=0.0005), 227 bias(decay=0.0)
Using 7187 train, 1797 val images for fraction=1.0 at imgsz=640
Using 12 dataloader workers
Logging results to /workspace/runs/detect/runs/optuna_4090/trial_0
Starting training for 15 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/15      1.27G     0.5422      5.758       1.05         15        640: 0% ──────────── 0/899  1.3s

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py:407: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


...

       2/15      11.4G     0.5978     0.6678    0.01058         30        640: 100% ━━━━━━━━━━━━ 225/225 11.1it/s 20.3s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 9.8it/s 3.0s<0.2s
                   all       1797       1820      0.959      0.931      0.979      0.905

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/15      11.4G     0.5755     0.5159   0.009917         51        640: 0% ──────────── 1/225 1.8it/s 0.2s<2:06

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1331: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  attn = ((q * self.scale).transpose(-2, -1) @ k).softmax(dim=-1)
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1332: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is 

       3/15      11.4G     0.5365     0.5432   0.009315         23        640: 100% ━━━━━━━━━━━━ 225/225 11.0it/s 20.4s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 9.5it/s 3.1s<0.1s
                   all       1797       1820      0.965      0.952      0.984      0.913

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/15      11.4G     0.5525     0.5154   0.009571         62        640: 0% ──────────── 1/225 1.7it/s 0.2s<2:10

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1331: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:233.)
  attn = ((q * self.scale).transpose(-2, -1) @ k).softmax(dim=-1)
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1332: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is 

       4/15      11.4G      0.525     0.5403   0.009203         48        640: 41% ━━━━╸─────── 93/225 11.3it/s 8.3s<11.6s

In [7]:
print(f"Trials recorded so far: {len(study.trials)}")
print(study.trials_dataframe()[["number", "state", "value", "params_model"]].to_string(index=False))

Trials recorded so far: 25
 number    state    value params_model
      0 COMPLETE 0.914659   yolov9t.pt
      1 COMPLETE 0.943316   yolo26n.pt
      2 COMPLETE 0.941266   yolov9s.pt
      3 COMPLETE 0.941034   yolov9t.pt
      4 COMPLETE 0.927298   yolo12n.pt
      5 COMPLETE 0.946500   yolo11s.pt
      6   PRUNED 0.853770   yolo12n.pt
      7   PRUNED 0.940710   yolo26n.pt
      8   PRUNED 0.934790   yolov8n.pt
      9 COMPLETE 0.945144   yolov8s.pt
     10   PRUNED 0.694260   yolo11s.pt
     11 COMPLETE 0.945571   yolo11s.pt
     12 COMPLETE 0.944281   yolo11s.pt
     13 COMPLETE 0.946660   yolo11s.pt
     14 COMPLETE 0.944733   yolo11s.pt
     15 COMPLETE 0.947460   yolo26s.pt
     16   PRUNED 0.885090   yolo11s.pt
     17   PRUNED 0.909580   yolo26s.pt
     18   PRUNED 0.899110   yolo26s.pt
     19   PRUNED 0.885920   yolo26s.pt
     20 COMPLETE 0.945366   yolo26n.pt
     21   PRUNED 0.885970   yolo11s.pt
     22 COMPLETE 0.950390   yolo26s.pt
     23 COMPLETE 0.949089   yolo26s.p

In [8]:
print("Best trial:", study.best_trial.number)
print("Params:", study.best_trial.params)
print("Extra info:", study.best_trial.user_attrs)

Best trial: 22
Params: {'model': 'yolo26s.pt', 'lr0': 0.00021847206846126456, 'batch': 32, 'mosaic': 0.23797477440943138}
Extra info: {'model_name': 'yolo26s.pt', 'map50': 0.989880838361551, 'inference_ms': 0.8604107686340743, 'trial_seconds': 368.9}


In [9]:
df = study.trials_dataframe()
df.to_csv("/workspace/optuna_screening_results_final.csv", index=False)

In [10]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

df = study.trials_dataframe()
df

,number,value,datetime_start,datetime_complete,duration,params_batch,params_lr0,params_model,params_mosaic,user_attrs_inference_ms,user_attrs_map50,user_attrs_model_name,user_attrs_trial_seconds,system_attrs_tpe:relative_params:0,state
0,0,0.914659,2026-10-14 05:27:08.629921,2026-10-14 05:39:25.558902,0 days 00:12:16.928981,8,0.008365,yolov9t.pt,0.892068,0.650147,0.973105,yolov9t.pt,736.7,NaN,COMPLETE
1,1,0.943316,2026-10-14 05:39:25.560727,2026-10-14 05:49:11.346943,0 days 00:09:45.786216,8,0.001114,yolo26n.pt,0.745800,0.475704,0.988021,yolo26n.pt,585.5,NaN,COMPLETE
2,2,0.941266,2026-10-14 05:49:11.348775,2026-10-14 06:01:47.692048,0 days 00:12:36.343273,8,0.000290,yolov9s.pt,0.724817,1.072815,0.986747,yolov9s.pt,756.1,NaN,COMPLETE
3,3,0.941034,2026-10-14 06:01:47.694802,2026-10-14 06:14:12.856653,0 days 00:12:25.161851,8,0.000118,yolov9t.pt,0.694296,0.538312,0.987132,yolov9t.pt,744.9,NaN,COMPLETE
4,4,0.927298,2026-10-14 06:14:12.859254,2026-10-14 06:19:13.338654,0 days 00:05:00.479400,32,0.001980,yolo12n.pt,0.882131,0.556146,0.980304,yolo12n.pt,300.2,NaN,COMPLETE
5,5,0.946500,2026-10-14 06:19:13.341252,2026-10-14 06:24:25.342773,0 days 00:05:12.001521,32,0.000156,yolo11s.pt,0.054924,0.892140,0.987088,yolo11s.pt,311.6,NaN,COMPLETE
6,6,0.853770,2026-10-14 06:24:25.349061,2026-10-14 06:26:15.734143,0 days 00:01:50.385082,32,0.000579,yolo12n.pt,0.364898,NaN,NaN,NaN,NaN,NaN,PRUNED
7,7,0.940710,2026-10-14 06:26:15.734870,2026-10-14 06:36:27.097733,0 days 00:10:11.362863,8,0.000226,yolo26n.pt,0.365736,NaN,NaN,NaN,NaN,NaN,PRUNED
8,8,0.934790,2026-10-14 06:36:27.098921,2026-10-14 06:38:48.010667,0 days 00:02:20.911746,32,0.000104,yolov8n.pt,0.088739,NaN,NaN,NaN,NaN,NaN,PRUNED
9,9,0.945144,2026-10-14 06:38:48.012001,2026-10-14 06:43:36.433490,0 days 00:04:48.421489,32,0.000275,yolov8s.pt,0.709711,0.852187,0.986406,yolov8s.pt,288.2,NaN,COMPLETE


In [11]:
!find / -maxdepth 7 -ipath "*optuna_4090*" -type d 2>/dev/null

/workspace/runs/detect/runs/optuna_4090
/workspace/runs/detect/runs/optuna_4090/trial_0
/workspace/runs/detect/runs/optuna_4090/trial_0/weights
/workspace/runs/detect/runs/optuna_4090/trial_1
/workspace/runs/detect/runs/optuna_4090/trial_1/weights
/workspace/runs/detect/runs/optuna_4090/trial_2
/workspace/runs/detect/runs/optuna_4090/trial_2/weights
/workspace/runs/detect/runs/optuna_4090/trial_3
/workspace/runs/detect/runs/optuna_4090/trial_3/weights
/workspace/runs/detect/runs/optuna_4090/trial_4
/workspace/runs/detect/runs/optuna_4090/trial_4/weights
/workspace/runs/detect/runs/optuna_4090/trial_5
/workspace/runs/detect/runs/optuna_4090/trial_5/weights
/workspace/runs/detect/runs/optuna_4090/trial_6
/workspace/runs/detect/runs/optuna_4090/trial_6/weights
/workspace/runs/detect/runs/optuna_4090/trial_7
/workspace/runs/detect/runs/optuna_4090/trial_7/weights
/workspace/runs/detect/runs/optuna_4090/trial_8
/workspace/runs/detect/runs/optuna_4090/trial_8/weights
/workspace/runs/detect/r

In [12]:
import os
import zipfile

SOURCE_DIR = "/workspace/runs/detect/runs/optuna_4090"
ZIP_PATH = "/workspace/optuna_4090_metrics_only.zip"

CSV_CANDIDATES = [
    "/workspace/optuna_screening_results.csv",
    "/workspace/optuna_screening_results_final.csv",
]

included, skipped = 0, 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # per-trial training logs, weights excluded
    for root, dirs, files in os.walk(SOURCE_DIR):
        for file in files:
            if file.endswith(".pt"):
                skipped += 1
                continue
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, SOURCE_DIR)
            zf.write(full_path, arcname)
            included += 1

    # the overall results CSV, at the root of the zip — add whichever exists
    for csv_path in CSV_CANDIDATES:
        if os.path.exists(csv_path):
            zf.write(csv_path, os.path.basename(csv_path))
            included += 1
            print(f"Added {csv_path}")

print(f"\nZipped {included} files, skipped {skipped} .pt weight files")
!du -h {ZIP_PATH}

Added /workspace/optuna_screening_results_final.csv

Zipped 51 files, skipped 50 .pt weight files
48K	/workspace/optuna_4090_metrics_only.zip


In [13]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
csv_in_zip = [n for n in names if n.endswith(".csv")]
print(f"{len(names)} files total")
print("CSV files in zip:", csv_in_zip)

51 files total
CSV files in zip: ['trial_0/results.csv', 'trial_1/results.csv', 'trial_2/results.csv', 'trial_3/results.csv', 'trial_4/results.csv', 'trial_5/results.csv', 'trial_6/results.csv', 'trial_7/results.csv', 'trial_8/results.csv', 'trial_9/results.csv', 'trial_10/results.csv', 'trial_11/results.csv', 'trial_12/results.csv', 'trial_13/results.csv', 'trial_14/results.csv', 'trial_15/results.csv', 'trial_16/results.csv', 'trial_17/results.csv', 'trial_18/results.csv', 'trial_19/results.csv', 'trial_20/results.csv', 'trial_21/results.csv', 'trial_22/results.csv', 'trial_23/results.csv', 'trial_24/results.csv', 'optuna_screening_results_final.csv']
